In [ ]:
from google.colab import files
uploaded = files.upload()

In [31]:
!pip install pytorch-lightning torchinfo torchmetrics torch-summary torchmetrics medmnist

In [32]:
import multiprocessing as mp
import os
import random
import sys
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pytorch_lightning as pl
import torch
import torchinfo
from medmnist import DermaMNIST
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy
from torchsummary import summary
from torchvision import datasets, models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

In [33]:
class EfeitosDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str = '/dataset', # Caminho para a pasta principal
        batch_size: int = 32,               # Tamanho do lote
        num_workers: int = 2,               # Threads para carregar os dados
        img_size: int = 256,                # Tamanho da imagem
        val_split: float = 0.2              # Porcentagem de dados para validação (20%)
    ):
        super().__init__()

        # Salva todos os argumentos do __init__ em self.hparams
        self.save_hyperparameters()

        # Define as transformações baseadas nos hiperparâmetros
        self.transform = transforms.Compose([
            transforms.Resize((self.hparams.img_size, self.hparams.img_size)),
            transforms.ToTensor()
        ])

    def setup(self, stage=None):
        """
        O setup é chamado em cada GPU (se você usar mais de uma).
        É aqui que dividimos o dataset em Treino e Validação/Teste.
        """
        dataset_completo = ImageFolder(root=self.hparams.data_dir, transform=self.transform)

        # Calcula a quantidade de imagens para treino e validação
        tamanho_total = len(dataset_completo)
        tamanho_val = int(self.hparams.val_split * tamanho_total)
        tamanho_treino = tamanho_total - tamanho_val

        # random_split garante que as imagens sejam separadas aleatoriamente
        self.dataset_treino, self.dataset_val = random_split(
            dataset_completo,
            [tamanho_treino, tamanho_val],
            generator=torch.Generator().manual_seed(42)
        )

    def train_dataloader(self):
        # Retorna o DataLoader de treino (com shuffle=True)
        return DataLoader(
            self.dataset_treino,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.num_workers,
            pin_memory=True # Acelera a transferência para a GPU no Colab
        )

    def val_dataloader(self):
        # Retorna o DataLoader de validação (com shuffle=False)
        return DataLoader(
            self.dataset_val,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=True
        )

In [39]:
# Instancia o DataModule passando os hiperparâmetros desejados
data_module = EfeitosDataModule(
    data_dir='/content/dataset',
    batch_size=16,
    img_size=224 # Por exemplo, mudando o tamanho da imagem
)

# Chama o setup manualmente (o Trainer do Lightning fará isso automaticamente depois)
data_module.setup()

# Pegando informações para conferir
print(f"Imagens para treino: {len(data_module.dataset_treino)}")
print(f"Imagens para validação: {len(data_module.dataset_val)}")

# Pegando um batch de validação para teste
dataloader_val = data_module.val_dataloader()
imagens, labels = next(iter(dataloader_val))
print(f"Shape do batch de imagens: {imagens.shape}")

FileNotFoundError: Found no valid file for the classes 297611_green_railing_dither, 297611_radial_vignette, origin. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [ ]:
class ClassificadorEfeitos(pl.LightningModule):
    def __init__(self, num_classes: int, lr: float = 1e-3):
        super().__init__()

        # Salva os hiperparâmetros (como a taxa de aprendizado e nº de classes)
        self.save_hyperparameters()

        # Carrega o modelo ResNet18 pré-treinado
        self.modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Substitui a última camada para corresponder ao nosso número de efeitos
        num_features_antigo = self.modelo.fc.in_features
        self.modelo.fc = nn.Linear(num_features_antigo, self.hparams.num_classes)

        # Define a função de perda (Loss) para classificação multiclasse
        self.criterio = nn.CrossEntropyLoss()

    def forward(self, x):
        # O forward é o que acontece quando passamos a imagem pela rede
        return self.modelo(x)

    def training_step(self, batch, batch_idx):
        imagens, labels = batch

        # Passa as imagens pelo modelo
        previsoes = self(imagens)

        # Calcula o erro (loss)
        loss = self.criterio(previsoes, labels)

        # Registra o erro para acompanharmos depois
        self.log('loss_treino', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        imagens, labels = batch
        previsoes = self(imagens)
        loss = self.criterio(previsoes, labels)

        # Calcula a acurácia (quantas acertou / total)
        preds = torch.argmax(previsoes, dim=1)
        acc = (preds == labels).float().mean()

        # Registra métricas de validação
        self.log('loss_val', loss, prog_bar=True)
        self.log('acc_val', acc, prog_bar=True)

    def configure_optimizers(self):
        # Define o otimizador (Adam é o padrão mais seguro)
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
NUM_EFEITOS = 5

#Instancia os Dados
data_module = EfeitosDataModule(
    data_dir='/dataset',
    batch_size=32,
    img_size=224
)

# Instancia o Modelo
modelo = ClassificadorEfeitos(num_classes=NUM_EFEITOS, lr=0.001)

# Configura o "Treinador" do Lightning
trainer = pl.Trainer(
    max_epochs=10,               # Quantas vezes ele vai ver o dataset inteiro
    accelerator='auto',          # Acha a GPU do Colab automaticamente se estiver ativa
    devices='auto',
    log_every_n_steps=10         # Atualiza a barra de progresso a cada 10 passos
)

trainer.fit(model=modelo, datamodule=data_module)

In [ ]:
class CNNEfeitos(pl.LightningModule):
    def __init__(self, num_classes: int, lr: float = 1e-3):
        super().__init__()

        # Salva a taxa de aprendizado e o número de classes nos hiperparâmetros
        self.save_hyperparameters()

        # Carrega a arquitetura ResNet-18 (sem Transfer Learning)
        # pesos aleatórios.
        self.modelo = models.resnet18(weights=None)

        num_features = self.modelo.fc.in_features
        self.modelo.fc = nn.Linear(num_features, self.hparams.num_classes)

        # Define a função de perda (Loss)
        self.criterio = nn.CrossEntropyLoss()

    def forward(self, x):
        # Define o fluxo da imagem através da rede
        return self.modelo(x)

    def training_step(self, batch, batch_idx):
        imagens, labels = batch

        # Passa as imagens pelo modelo para obter as previsões brutas (logits)
        previsoes = self(imagens)

        # Calcula o erro (loss)
        loss = self.criterio(previsoes, labels)

        # Registra a loss de treino para acompanharmos no gráfico
        self.log('loss_treino', loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        imagens, labels = batch

        # No passo de validação, fazemos a mesma coisa, mas sem atualizar os pesos
        previsoes = self(imagens)
        loss = self.criterio(previsoes, labels)

        # Calcula a acurácia (quantas imagens a rede acertou o efeito)
        # torch.argmax pega o índice do efeito com maior probabilidade
        preds = torch.argmax(previsoes, dim=1)
        acc = (preds == labels).float().mean()

        # Registra as métricas de validação
        self.log('loss_val', loss, prog_bar=True, on_epoch=True)
        self.log('acc_val', acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        # Define o otimizador Adam
        # Ele é o responsável por ajustar os pesos da rede com base na loss
        otimizador = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return otimizador

In [ ]:
# --- Configurações iniciais ---
NUMERO_DE_EFEITOS = 5 # Substitua pela quantidade exata de pastas/efeitos que você tem

data_module = EfeitosDataModule(
    data_dir='/dataset',
    batch_size=32,
    img_size=224 # A ResNet foi desenhada para imagens 224x224
)

# Instancia o seu Modelo do Zero
modelo = CNNEfeitos(num_classes=NUMERO_DE_EFEITOS, lr=0.001)

#Configura o Treinador (Trainer)
trainer = pl.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices='auto',
    log_every_n_steps=10
)

# 4. Inicia o Treinamento
trainer.fit(model=modelo, datamodule=data_module)

In [ ]:
def extrair_dados_para_ml(dataloader):
    imagens_achatadas = []
    labels_lista = []

    print("Extraindo imagens do DataLoader...")
    for imagens, labels in dataloader:
        # imagens original: [Batch, 3, 256, 256] (ou 224x224)
        batch_size = imagens.size(0)

        # Achata a imagem: O -1 faz o PyTorch calcular automaticamente o tamanho final
        imagens_flat = imagens.view(batch_size, -1).numpy()

        imagens_achatadas.append(imagens_flat)
        labels_lista.append(labels.numpy())

    # Junta todas as listas em uma matriz única do NumPy
    X = np.vstack(imagens_achatadas)
    y = np.concatenate(labels_lista)

    return X, y

# Pegamos o dataloader de treino do seu DataModule (que criamos lá atrás)
dataloader_treino = data_module.train_dataloader()
X_treino, y_treino = extrair_dados_para_ml(dataloader_treino)

print(f"Formato de X_treino: {X_treino.shape}")
print(f"Formato de y_treino: {y_treino.shape}")

In [ ]:
print("Calculando o PCA (Isso pode levar alguns segundos/minutos dependendo da RAM)...")

# Instancia o PCA pedindo 50 componentes
pca = PCA(n_components=50, random_state=42)

# O fit_transform faz duas coisas:
# 1. Encontra os padrões (fit)
# 2. Aplica a redução nos dados (transform)
X_treino_pca = pca.fit_transform(X_treino)

print(f"Formato após o PCA: {X_treino_pca.shape}")